# 08 · Profiling — objective ADI / CV² threshold search + business overlays

Notebook 03 classifies every series into the Syntetos–Boylan quadrants with the textbook cut-offs
`ADI = 1.32` and `CV² = 0.49`. Those numbers were derived for **retail demand in units**; on durable
goods (project deals, spare parts, service contracts, revenue rather than units) they are a guess.
This notebook turns the thresholds into a **decision with an objective**:

1. compute the SBC indicators once per series (`ADI` from order months, `CV²` of the non-zero demand sizes, `SDDI`);
2. backtest a small set of deliberately simple **candidate methods** once per series (threshold-independent) —
   one per quadrant: seasonal naive for `smooth`, a robust 12-month median for `erratic`, a Croston-style demand
   rate for `intermittent` / `lumpy`, last value / zero for the degenerate profiles;
3. run a **randomized search** over `(thres_cv2, thres_adi)` — the textbook pair is trial 0 — and, for every
   candidate, label the series and **route each series to its quadrant's method**; the objective is the pooled
   WMAPE of the whole portfolio under that routing. This is what the classification is *for*: picking the right
   method family per series. Scoring only one model on one group leaves the CV² threshold unidentified;
4. keep the thresholds with the lowest WMAPE, **subject to a guardrail** — the regular group (`smooth` + `erratic`)
   must still cover a minimum share of revenue, so the optimiser cannot "win" by declaring almost everything
   sparse and forecasting flat rates;
5. overlay **business clusters** on top of the SBC profile (a dominant *strategic* series, *possibly obsolete*
   and *new* series) with an explicit priority, and backtest every group so the modelling strategy per group
   is chosen on evidence.

Because the panel is synthetic, the last section compares the recovered labels with the generator's ground
truth — a check you will not have in production, but a useful calibration of what a threshold search can and
cannot recover.

| | |
|---|---|
| **Reads** | `<scenario>_prepared` (notebook 07) — optional: `<scenario>_series_master` for the synthetic-only check |
| **Writes** | `<scenario>_profiled` (+ `profile`), `<scenario>_clustered` (+ `profile`, `profile_cluster`) |
| **Profiled on** | `quantity` sizes (CV²) and `orders` occurrences (ADI) — *what drives the pattern* |
| **Scored on** | `y` (net revenue) — *what the business is held to* |

## 1. Imports

In [ ]:
import os
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import uniform
from scipy.stats.mstats import winsorize
from sklearn.model_selection import ParameterSampler

print("✅ Libraries imported")

## 2. Configuration

In [ ]:
SCENARIO = "durable_goods_monthly"
unique_id, date_var, y = "unique_id", "ds", "y"
QTY_COL, ORDERS_COL = "quantity", "orders"          # CV² on quantity sizes, ADI on order months
HIER_COLS = ["product_family", "product_group", "product_line", "region", "channel"]

SEASON = 12            # monthly seasonality
HORIZON = 6            # months held out for the simple-model backtest
MIN_DEMAND = 3         # <= this many non-zero months -> constant_zero
CV2_CONST = 0.0032     # below this CV² -> constant
WINSOR = 0.05          # winsorisation of the non-zero sizes before CV²

# search space (ParameterSampler) + guardrails
N_ITER = 60
RANDOM_STATE = 42
CV2_RANGE = (0.10, 1.20)     # thres_cv2 ~ U(0.10, 1.20)
ADI_RANGE = (1.05, 2.50)     # thres_adi ~ U(1.05, 2.50)
REGULAR = ("smooth", "erratic")
MIN_REVENUE_SHARE = 0.40     # the regular group must carry >= 40% of revenue ...
MIN_SERIES = 40              # ... and >= 40 series, else the candidate is infeasible
# the simple method each quadrant is routed to (threshold-independent forecasts, computed once per series)
METHOD_BY_PROFILE = {"smooth": "snaive", "erratic": "median12", "intermittent": "rate12", "lumpy": "rate24",
                     "constant": "last", "constant_zero": "zero"}

# business overlays (priority: strategic > possible_obsolete > new_expansion > SBC profile)
STRATEGIC_MIN_SHARE = 0.08   # a single series above 8% of total revenue gets its own cluster
RECENT_MONTHS = 12

# IO (Lakehouse first, local parquet fallback)
LAKEHOUSE_NAME = "ts_forecasting"
LOCAL_FOLDER = SCENARIO
root_path = Path.cwd().parent.parent
local_dir = root_path / "data" / LOCAL_FOLDER
PREPARED_TABLE, MASTER_TABLE = f"{SCENARIO}_prepared", f"{SCENARIO}_series_master"
PROFILED_TABLE, CLUSTERED_TABLE = f"{SCENARIO}_profiled", f"{SCENARIO}_clustered"

# optional MLflow tracking (on when a tracking URI is configured or when running on a platform)
MLFLOW_ENABLED = bool(os.environ.get("MLFLOW_TRACKING_URI")) or os.path.isdir("/lakehouse") or bool(os.environ.get("DATABRICKS_RUNTIME_VERSION"))
EXPERIMENT = f"/Shared/{SCENARIO}_profiling"


def load_table(name: str) -> pd.DataFrame:
    try:
        df = spark.table(f"{LAKEHOUSE_NAME}.{name}").toPandas()
        print(f"✅ Loaded {LAKEHOUSE_NAME}.{name} ({len(df):,} rows)")
    except Exception as e:
        df = pd.read_parquet(local_dir / f"{name}.parquet")
        print(f"ℹ️ Spark unavailable ({type(e).__name__}); loaded {local_dir / (name + '.parquet')} ({len(df):,} rows)")
    return df


def save_table(df: pd.DataFrame, name: str) -> None:
    try:
        spark.createDataFrame(df).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{LAKEHOUSE_NAME}.{name}")
        print(f"✅ Saved {len(df):,} rows to {LAKEHOUSE_NAME}.{name}")
    except Exception as e:
        local_dir.mkdir(parents=True, exist_ok=True)
        df.to_parquet(local_dir / f"{name}.parquet", index=False)
        print(f"ℹ️ Spark unavailable ({type(e).__name__}); saved {len(df):,} rows to {local_dir / (name + '.parquet')}")


class _MLF:
    """No-op MLflow facade so the notebook runs identically with or without tracking."""
    def __init__(self, enabled):
        self.enabled = enabled
        if enabled:
            import mlflow
            self.m = mlflow
            try:
                mlflow.set_experiment(EXPERIMENT)
            except Exception as e:
                print(f"⚠️ set_experiment failed ({e}); using the default experiment")
    def start(self, name, nested=False):
        return self.m.start_run(run_name=name, nested=nested) if self.enabled else _Null()
    def log_params(self, d):
        if self.enabled: self.m.log_params(d)
    def log_metrics(self, d):
        if self.enabled: self.m.log_metrics({k: float(v) for k, v in d.items() if v == v})
    def log_figure(self, fig, path):
        if self.enabled: self.m.log_figure(fig, path)


class _Null:
    def __enter__(self): return self
    def __exit__(self, *a): return False


mlf = _MLF(MLFLOW_ENABLED)
print(f"✅ Config | target={y} | profile on {QTY_COL}/{ORDERS_COL} | horizon={HORIZON}m | MLflow={'on' if MLFLOW_ENABLED else 'off'}")

## 3. Load the prepared panel

In [ ]:
df = load_table(PREPARED_TABLE)
df[date_var] = pd.to_datetime(df[date_var])
for c in (y, QTY_COL, ORDERS_COL):
    df[c] = df[c].fillna(0.0).astype(float)
df = df.sort_values([unique_id, date_var]).reset_index(drop=True)
try:
    master = load_table(MASTER_TABLE)
except Exception:
    master = None
DATA_MAX = df[date_var].max()
print(f"rows {len(df):,} | series {df[unique_id].nunique()} | {df[date_var].min().date()} -> {DATA_MAX.date()}")

## 4. SBC indicators (computed once — they do not depend on the thresholds)

- **ADI** = periods ÷ periods with at least one order (demand *sparsity*).
- **CV²** = squared coefficient of variation of the winsorised non-zero quantities (size *volatility*).
- **SDDI** = standard deviation of the inter-order interval (regularity of the gaps; reported, not used
  in the quadrant rule — see notebook 03 for the enhanced classification that uses it).

In [ ]:
def per_series_arrays(frame, col):
    return {uid: s.to_numpy(float) for uid, s in frame.groupby(unique_id, sort=False)[col]}


def sbc_indicators(qty, orders, winsor=WINSOR):
    n = int(qty.size)
    order_pos = np.flatnonzero(orders > 0) if orders is not None and (orders > 0).any() else np.flatnonzero(qty > 0)
    qty_pos = np.flatnonzero(qty > 0)
    if order_pos.size == 0 or qty_pos.size == 0:
        return {"n_periods": n, "n_demand": int(qty_pos.size), "adi": np.inf, "cv2": np.nan, "sddi": np.nan}
    nz = winsorize(qty[qty_pos], (winsor, winsor)) if qty_pos.size > 4 else qty[qty_pos]
    mean_nz = float(np.mean(nz))
    cv2 = float((np.std(nz) / mean_nz) ** 2) if mean_nz else np.nan
    gaps = np.diff(order_pos)
    return {"n_periods": n, "n_demand": int(qty_pos.size), "adi": n / order_pos.size, "cv2": cv2,
            "sddi": float(np.std(gaps)) if gaps.size else 0.0}


def classify(ind, thres_cv2, thres_adi):
    if ind["n_demand"] <= MIN_DEMAND or not np.isfinite(ind["cv2"]):
        return "constant_zero"
    low_adi = ind["adi"] < thres_adi
    if ind["cv2"] < CV2_CONST and low_adi:      # a sparse series whose sizes are always 1 unit has CV²=0 but is NOT constant
        return "constant"
    return {(True, True): "smooth", (False, True): "intermittent",
            (True, False): "erratic", (False, False): "lumpy"}[(low_adi, ind["cv2"] < thres_cv2)]


qty_arrays, ord_arrays, y_arrays = per_series_arrays(df, QTY_COL), per_series_arrays(df, ORDERS_COL), per_series_arrays(df, y)
ind_records = {uid: sbc_indicators(qty_arrays[uid], ord_arrays.get(uid)) for uid in qty_arrays}
ind_df = pd.DataFrame(ind_records).T.rename_axis(unique_id).reset_index()
print(f"indicators for {len(ind_df):,} series")
ind_df[["adi", "cv2", "sddi", "n_demand"]].astype(float).describe().round(3)

## 5. Candidate simple methods, backtested once per series

The objective must not depend on a sophisticated model — otherwise the thresholds would be tuned to that
model's quirks. Each quadrant gets the simplest method that fits its story, forecast over the last `HORIZON`
months:

| profile | method | idea |
|---|---|---|
| `smooth` | `snaive` | same month last year (falls back to the last value) |
| `erratic` | `median12` | robust level: median of the last 12 months — spikes should not be repeated |
| `intermittent` | `rate12` | Croston-style demand rate: mean of the last 12 months **including zeros** |
| `lumpy` | `rate24` | the same rate over 24 months — rare big deals need a longer window |
| `constant` / `constant_zero` | `last` / `zero` | |

Every candidate is computed for every series once; a threshold pair only decides **which** candidate each
series uses. Errors are pooled per group as **WMAPE** (zero-safe), with **RMSE** and the **median MASE**
(scaled by the in-sample seasonal-naive error) alongside.

In [ ]:
def candidate_forecasts(train, h, m=SEASON):
    train = np.asarray(train, float)
    last = train[-1] if train.size else 0.0
    snaive = np.tile(train[-m:], int(math.ceil(h / m)))[:h] if train.size >= m else np.full(h, last)
    return {"snaive": snaive,
            "median12": np.full(h, float(np.median(train[-12:]))),
            "rate12": np.full(h, float(np.mean(train[-12:]))),
            "rate24": np.full(h, float(np.mean(train[-24:]))),
            "last": np.full(h, last),
            "zero": np.zeros(h)}


def backtest_series(arr, h=HORIZON, m=SEASON):
    arr = np.asarray(arr, float)
    if arr.size <= max(h + 1, m + h):
        return None
    train, test = arr[:-h], arr[-h:]
    preds = candidate_forecasts(train, h, m)
    scale = np.mean(np.abs(train[m:] - train[:-m])) if train.size > m else np.nan
    out = {"act": float(np.abs(test).sum()), "n": h, "scale": float(scale) if scale and scale > 0 else np.nan,
           "test": test, "pred": preds, "abs_err": {}, "sq_err": {}, "mae": {}}
    for k, p in preds.items():
        e = np.abs(test - p)
        out["abs_err"][k], out["sq_err"][k], out["mae"][k] = float(e.sum()), float((e ** 2).sum()), float(e.mean())
    return out


def aggregate_metrics(bt, routing):
    """Pool the error of each series under its routed method. routing: {uid: method}."""
    rows = [(bt[u], meth) for u, meth in routing.items() if u in bt]
    if not rows:
        return {"wmape": np.nan, "rmse": np.nan, "mase_med": np.nan, "n_series": 0}
    abs_err = sum(r["abs_err"][k] for r, k in rows)
    act = sum(r["act"] for r, _ in rows)
    sq, n = sum(r["sq_err"][k] for r, k in rows), sum(r["n"] for r, _ in rows)
    mases = [r["mae"][k] / r["scale"] for r, k in rows if r["scale"] == r["scale"]]
    return {"wmape": 100.0 * abs_err / act if act else np.nan, "rmse": math.sqrt(sq / n) if n else np.nan,
            "mase_med": float(np.median(mases)) if mases else np.nan, "n_series": len(rows)}


bt = {uid: r for uid, a in y_arrays.items() if (r := backtest_series(a)) is not None}
rev_by_uid = df.groupby(unique_id)[y].sum()
TOTAL_REV = float(rev_by_uid.sum())
# routing oracle: the per-series best candidate - the floor no threshold pair can beat
oracle_routing = {u: min(r["abs_err"], key=r["abs_err"].get) for u, r in bt.items()}
ORACLE = aggregate_metrics(bt, oracle_routing)
print(f"backtestable series: {len(bt):,} | horizon {HORIZON} months | objective = portfolio WMAPE on {y} under quadrant routing")
print(f"routing oracle (best method per series, not achievable): WMAPE {ORACLE['wmape']:.1f}%")
print("single-method baselines (every series on one method): " + ", ".join(
    f"{k} {aggregate_metrics(bt, {u: k for u in bt})['wmape']:.1f}%" for k in ["snaive", "median12", "rate12", "rate24"]))

## 6. Randomized threshold search with a guardrail

Trial 0 is the textbook pair `(0.49, 1.32)`. A candidate is **infeasible** when its regular group
(`smooth` + `erratic`) falls below `MIN_REVENUE_SHARE` of revenue or `MIN_SERIES` series — without it the
optimiser can drift towards "call everything sparse" whenever flat rates happen to score well on the hold-out.

In [ ]:
def evaluate(thres_cv2, thres_adi):
    labels = {uid: classify(ind, thres_cv2, thres_adi) for uid, ind in ind_records.items()}
    routing = {uid: METHOD_BY_PROFILE[lab] for uid, lab in labels.items()}
    m = aggregate_metrics(bt, routing)
    regular = [uid for uid, lab in labels.items() if lab in REGULAR]
    m["rev_share"] = float(rev_by_uid.reindex(regular).sum() / TOTAL_REV) if TOTAL_REV else np.nan
    m["n_regular"] = len(regular)
    m["feasible"] = (m["rev_share"] >= MIN_REVENUE_SHARE) and (len(regular) >= MIN_SERIES)
    m["oracle_agreement"] = float(np.mean([routing[u] == oracle_routing[u] for u in bt]))
    counts = pd.Series(list(labels.values())).value_counts().to_dict()
    return labels, routing, m, counts


def per_profile_metrics(labels, routing):
    out = {}
    for p in sorted(set(labels.values())):
        ids = {u: routing[u] for u, lab in labels.items() if lab == p}
        out[p] = aggregate_metrics(bt, ids)
    return out


space = {"thres_cv2": uniform(CV2_RANGE[0], CV2_RANGE[1] - CV2_RANGE[0]),
         "thres_adi": uniform(ADI_RANGE[0], ADI_RANGE[1] - ADI_RANGE[0])}
samples = [{"thres_cv2": 0.49, "thres_adi": 1.32}] + list(ParameterSampler(space, n_iter=N_ITER, random_state=RANDOM_STATE))

results = []
with mlf.start("threshold_search"):
    mlf.log_params({"target": y, "horizon": HORIZON, "season": SEASON, "n_iter": N_ITER,
                    "min_revenue_share": MIN_REVENUE_SHARE, "min_series": MIN_SERIES, "routing": str(METHOD_BY_PROFILE)})
    for i, s in enumerate(samples):
        labels, routing, m, counts = evaluate(**s)
        with mlf.start("textbook_0.49_1.32" if i == 0 else f"trial_{i}", nested=True):
            mlf.log_params({k: round(v, 4) for k, v in s.items()})
            mlf.log_metrics({"wmape": m["wmape"], "rmse": m["rmse"], "mase_med": m["mase_med"],
                             "rev_share_regular": m["rev_share"], "n_regular": m["n_regular"]})
            for p, pm in per_profile_metrics(labels, routing).items():
                mlf.log_metrics({f"wmape_{p}": pm["wmape"], f"n_series_{p}": pm["n_series"]})
        results.append({**s, **m, "n_smooth": counts.get("smooth", 0), "n_erratic": counts.get("erratic", 0),
                        "n_intermittent": counts.get("intermittent", 0), "n_lumpy": counts.get("lumpy", 0)})
    results_df = pd.DataFrame(results)
    feasible = results_df[results_df["feasible"]].sort_values("wmape")
    if feasible.empty:
        raise RuntimeError("no feasible candidate - relax MIN_REVENUE_SHARE / MIN_SERIES or widen the search space")
    best = feasible.iloc[0]
    BEST_CV2, BEST_ADI = float(best["thres_cv2"]), float(best["thres_adi"])
    mlf.log_params({"best_thres_cv2": round(BEST_CV2, 4), "best_thres_adi": round(BEST_ADI, 4)})
    mlf.log_metrics({"best_wmape": best["wmape"], "best_rev_share": best["rev_share"], "oracle_wmape": ORACLE["wmape"]})

textbook = results_df.iloc[0]
print(f"textbook (0.49, 1.32): portfolio WMAPE {textbook['wmape']:.1f}% | regular group {textbook['rev_share']:.0%} of revenue, "
      f"{int(textbook['n_regular'])} series | routes {textbook['oracle_agreement']:.0%} of series to their best method | feasible={textbook['feasible']}")
print(f"🏆 best feasible: CV2={BEST_CV2:.3f}, ADI={BEST_ADI:.3f} | portfolio WMAPE {best['wmape']:.1f}% | regular group "
      f"{best['rev_share']:.0%} of revenue, {int(best['n_regular'])} series | oracle agreement {best['oracle_agreement']:.0%}")
unconstrained = results_df.sort_values("wmape").iloc[0]
print(f"   unconstrained optimum: CV2={unconstrained['thres_cv2']:.2f}, ADI={unconstrained['thres_adi']:.2f} -> WMAPE {unconstrained['wmape']:.1f}% "
      f"with {unconstrained['rev_share']:.0%} of revenue in the regular group -> "
      f"{'rejected by the guardrail' if not unconstrained['feasible'] else 'feasible (same as best)'}")
print(f"   routing oracle floor: {ORACLE['wmape']:.1f}%")
feasible.head(8)[["thres_cv2", "thres_adi", "wmape", "rmse", "mase_med", "rev_share", "n_regular", "oracle_agreement",
                  "n_smooth", "n_erratic", "n_intermittent", "n_lumpy"]].round(3)

### The search landscape and the guardrail

Left: every trial in threshold space coloured by portfolio WMAPE (infeasible trials hollow). Right: the
accuracy-vs-coverage view — the guardrail line is where the business requirement ("this much revenue must be
modelled as regular demand") cuts the search; read the minimum to the right of it, not the global minimum.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
ax = axes[0]
f_ok, f_no = results_df[results_df["feasible"]], results_df[~results_df["feasible"]]
sc = ax.scatter(f_ok["thres_cv2"], f_ok["thres_adi"], c=f_ok["wmape"], cmap="viridis_r", s=70, label="feasible")
ax.scatter(f_no["thres_cv2"], f_no["thres_adi"], facecolor="none", edgecolor="grey", s=70, label="infeasible")
ax.scatter([0.49], [1.32], marker="s", s=120, facecolor="none", edgecolor="red", linewidth=2, label="textbook")
ax.scatter([BEST_CV2], [BEST_ADI], marker="*", s=360, facecolor="gold", edgecolor="k", label="best feasible", zorder=5)
ax.set(xlabel="thres_cv2", ylabel="thres_adi", title="Randomized search - WMAPE landscape")
fig.colorbar(sc, ax=ax, label="WMAPE (%)")
ax.legend(fontsize=8)
ax = axes[1]
ax.scatter(results_df["rev_share"] * 100, results_df["wmape"], c=np.where(results_df["feasible"], "steelblue", "lightgrey"), s=50)
ax.axvline(MIN_REVENUE_SHARE * 100, color="red", linestyle="--", label=f"min revenue share {MIN_REVENUE_SHARE:.0%}")
ax.scatter([best["rev_share"] * 100], [best["wmape"]], marker="*", s=360, facecolor="gold", edgecolor="k", zorder=5)
ax.set(xlabel="revenue covered by the regular group (%)", ylabel="portfolio WMAPE under routing (%)", title="Accuracy vs coverage trade-off")
ax.legend(fontsize=8)
fig.tight_layout()
mlf.log_figure(fig, "search/threshold_landscape.png")
plt.show()

## 7. Assign the per-series profile and look at the quadrants

In [ ]:
best_labels, best_routing, best_m, best_counts = evaluate(BEST_CV2, BEST_ADI)
prof = ind_df.copy()
prof["profile"] = prof[unique_id].map(best_labels)
prof["thres_cv2"], prof["thres_adi"] = BEST_CV2, BEST_ADI
prof = prof.merge(df[[unique_id, *HIER_COLS]].drop_duplicates(unique_id), on=unique_id, how="left")
prof["revenue_share"] = prof[unique_id].map(rev_by_uid) / TOTAL_REV
print("profile counts:\n" + prof["profile"].value_counts().to_string())
print("\nbacktest per profile under the chosen thresholds (each profile on its own method):")
print(pd.DataFrame(per_profile_metrics(best_labels, best_routing)).T[["n_series", "wmape", "mase_med"]].round(2).to_string())

fig, ax = plt.subplots(figsize=(8, 5.5))
pal = {"smooth": "#2ca02c", "erratic": "#d62728", "intermittent": "#1f77b4", "lumpy": "#9467bd",
       "constant": "#7f7f7f", "constant_zero": "#bcbd22"}
pp = prof[np.isfinite(prof["adi"].astype(float)) & prof["cv2"].notna()]
for p, g in pp.groupby("profile"):
    ax.scatter(g["adi"].astype(float), g["cv2"].astype(float), s=18 + 600 * g["revenue_share"], alpha=0.65, color=pal.get(p, "k"), label=p)
ax.axvline(BEST_ADI, color="k", linestyle="--", linewidth=1)
ax.axhline(BEST_CV2, color="k", linestyle="--", linewidth=1)
ax.axvline(1.32, color="red", linestyle=":", linewidth=1)
ax.axhline(0.49, color="red", linestyle=":", linewidth=1)
ax.set(xscale="log", yscale="log", xlabel="ADI (log)", ylabel="CV² (log)",
       title=f"SBC quadrants - chosen cut-offs (black) vs textbook (red); marker size = revenue share")
ax.legend(fontsize=8, markerscale=0.8)
fig.tight_layout()
mlf.log_figure(fig, "best/quadrants.png")
plt.show()

### Synthetic-only check: how much of the generator's truth did the search recover?

In production there is no ground truth. Here the series master tells us the archetype each series was
generated from, so we can see the confusion between *true* and *recovered* labels — expect frequent
`smooth ↔ erratic` swaps (they share the ADI side and differ only in size volatility) and `intermittent ↔ lumpy`
swaps; the ADI side is recovered almost perfectly.

In [ ]:
if master is not None and "archetype_true" in master.columns:
    cmp = prof.merge(master[[unique_id, "archetype_true"]], on=unique_id)
    cmp["side_true"] = np.where(cmp["archetype_true"].isin(["smooth", "erratic", "constant"]), "frequent", "sparse")
    cmp["side_rec"] = np.where(cmp["profile"].isin(["smooth", "erratic", "constant"]), "frequent", "sparse")
    print(f"exact label agreement : {(cmp['archetype_true'] == cmp['profile']).mean():.0%}")
    print(f"ADI side agreement    : {(cmp['side_true'] == cmp['side_rec']).mean():.0%}  (frequent vs sparse)")
    print(pd.crosstab(cmp["archetype_true"], cmp["profile"]).to_string())
else:
    print("series master not available - skipping the synthetic ground-truth check")

## 8. Business overlays → `profile_cluster`

The SBC profile describes the *shape*; three business facts override it, in this priority:

1. **`strategic`** — any single series above `STRATEGIC_MIN_SHARE` of total revenue is modelled on its own
   (pooling it with hundreds of small series lets the pool dilute the one number leadership looks at).
2. **`possible_obsolete`** — no quantity, orders or revenue in the last `RECENT_MONTHS` but activity before.
3. **`new_expansion`** — activity only inside the last `RECENT_MONTHS` (too short a history for lags of 12).

Everything else keeps its SBC profile. The column is called `profile_cluster` to match the contract that
notebook 06 and every skill expect.

In [ ]:
clus = prof[[unique_id, "profile", "revenue_share"]].copy()
active = (df[QTY_COL] > 0) | (df[ORDERS_COL] > 0) | (df[y] != 0)
cutoff = DATA_MAX - pd.DateOffset(months=RECENT_MONTHS - 1)
recent = df[date_var] >= cutoff
act = pd.DataFrame({unique_id: df[unique_id], "recent_active": active & recent, "older_active": active & ~recent}) \
    .groupby(unique_id).any().reset_index()
clus = clus.merge(act, on=unique_id, how="left").fillna({"recent_active": False, "older_active": False})

is_strategic = clus["revenue_share"] >= STRATEGIC_MIN_SHARE
is_obsolete = (~clus["recent_active"]) & clus["older_active"]
is_new = clus["recent_active"] & (~clus["older_active"])
clus["profile_cluster"] = clus["profile"]
clus.loc[is_new, "profile_cluster"] = "new_expansion"
clus.loc[is_obsolete, "profile_cluster"] = "possible_obsolete"
clus.loc[is_strategic, "profile_cluster"] = "strategic"          # highest priority applied last

print(f"recent window {cutoff.date()} -> {DATA_MAX.date()}")
print(f"strategic: {int(is_strategic.sum())} ({', '.join(clus.loc[is_strategic, unique_id])}) | "
      f"possible_obsolete: {int(is_obsolete.sum())} | new_expansion: {int(is_new.sum())}")
print("\nprofile_cluster counts:\n" + clus["profile_cluster"].value_counts().to_string())

## 9. Backtest by cluster group — the evidence for the modelling strategy

The same simple-method backtest, pooled per `profile_cluster` (every series on its profile's method; the
business overlays on the seasonal naive). Read it as a *floor*: a group where the simple method already scores
well is a candidate for the pooled LightGBM quantile models (notebook 09); a group where it does not is a
candidate for the Croston family (intermittent skill) or for a seasonal-naive fallback. Note how differently the
groups behave on **WMAPE** (dollars) and **MASE** (skill vs one-step naive), and two artefacts to expect:
`possible_obsolete` has no actuals in the hold-out (WMAPE undefined), and the MASE of `constant_zero` /
`new_expansion` explodes because their in-sample naive error (the denominator) is ~0 — which is why the advanced
track always reports the **median** MASE and a revenue-weighted variant, never the mean.

In [ ]:
clus["method"] = clus["profile"].map(METHOD_BY_PROFILE)
clus.loc[~clus["profile_cluster"].isin(METHOD_BY_PROFILE), "method"] = "snaive"      # overlays: seasonal naive
cluster_ids = {g: sub[unique_id].tolist() for g, sub in clus.groupby("profile_cluster")}
cluster_routing = {g: dict(zip(sub[unique_id], sub["method"])) for g, sub in clus.groupby("profile_cluster")}
cluster_metrics = (pd.DataFrame({g: aggregate_metrics(bt, r) for g, r in cluster_routing.items()}).T
                   .assign(rev_share=lambda t: [rev_by_uid.reindex(cluster_ids[g]).sum() / TOTAL_REV for g in t.index],
                           method=lambda t: [clus.loc[clus["profile_cluster"] == g, "method"].mode().iloc[0] for g in t.index])
                   [["n_series", "rev_share", "method", "wmape", "rmse", "mase_med"]].sort_values("rev_share", ascending=False))
with mlf.start("cluster_backtest"):
    for g, row in cluster_metrics.iterrows():
        mlf.log_metrics({f"wmape_{g}": row["wmape"], f"mase_med_{g}": row["mase_med"], f"n_series_{g}": row["n_series"]})
print(cluster_metrics.round(3).to_string())

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
cluster_metrics["wmape"].plot(kind="bar", ax=axs[0], color="indianred", rot=45, title="simple-method WMAPE (%) by profile_cluster")
cluster_metrics["mase_med"].plot(kind="bar", ax=axs[1], color="steelblue", rot=45, title="median MASE by profile_cluster")
axs[1].axhline(1.0, color="k", linewidth=1, linestyle="--")
axs[1].set_yscale("log")
fig.tight_layout()
mlf.log_figure(fig, "clusters/error_rates.png")
plt.show()

In [ ]:
SAMPLES_PER_GROUP = 2
random.seed(RANDOM_STATE)
groups = [g for g in cluster_metrics.index if g in cluster_ids]
fig, axes = plt.subplots(len(groups), SAMPLES_PER_GROUP, figsize=(13, 2.3 * len(groups)), squeeze=False)
for i, g in enumerate(groups):
    pool = [u for u in cluster_ids[g] if u in bt]
    pick = random.sample(pool, min(SAMPLES_PER_GROUP, len(pool))) if pool else []
    for j in range(SAMPLES_PER_GROUP):
        ax = axes[i, j]
        if j >= len(pick):
            ax.axis("off"); continue
        uid = pick[j]
        meth = cluster_routing[g][uid]
        hist = y_arrays[uid] / 1e3
        idx = np.arange(hist.size)
        ax.plot(idx, hist, color="0.45", label="actual")
        ax.plot(idx[-HORIZON:], bt[uid]["pred"][meth] / 1e3, "r--o", ms=3, label="routed simple method")
        ax.axvspan(idx[-HORIZON] - 0.5, idx[-1] + 0.5, color="orange", alpha=0.10)
        ax.set_title(f"{g} | {uid} | {meth} (ADI {ind_records[uid]['adi']:.2f}, CV² {ind_records[uid]['cv2']:.2f})", fontsize=8, loc="left")
        ax.set_ylabel("k")
        if i == 0 and j == 0:
            ax.legend(fontsize=7)
fig.suptitle("Simple-model backtest - sample series per profile_cluster", y=1.0)
fig.tight_layout()
plt.show()

## 10. Write `_profiled` and `_clustered` as extensions of the prepared panel

In [ ]:
profiled = df.merge(prof[[unique_id, "profile", "adi", "cv2", "sddi"]], on=unique_id, how="left")
clustered = profiled.merge(clus[[unique_id, "profile_cluster"]], on=unique_id, how="left")
assert clustered[["profile", "profile_cluster"]].notna().all().all(), "every row must carry a profile and a profile_cluster"
save_table(profiled, PROFILED_TABLE)
save_table(clustered, CLUSTERED_TABLE)
print(f"➕ added to _profiled: profile, adi, cv2, sddi | ➕ added to _clustered: profile_cluster")

## ✅ CHECK POINT with the data scientist

- Thresholds chosen: `thres_cv2 = {BEST_CV2}`, `thres_adi = {BEST_ADI}` (printed above) with the guardrail
  `MIN_REVENUE_SHARE` / `MIN_SERIES`. If the business needs a *larger* forecastable group, raise the guardrail
  and re-run — the landscape chart shows the price in WMAPE.
- The quadrant chart is the review artefact: confirm that the biggest bubbles (revenue) sit in the group you
  intend to model with the quantile models.
- `profile_cluster` carries the business overlays; `strategic` and `smooth` / `erratic` are the candidates for
  [09 · Quantile blend walk-forward](09-Quantile-Blend-Walk-Forward); `intermittent` / `lumpy` go to the
  Croston-family benchmark; `possible_obsolete` / `new_expansion` / `constant*` default to the seasonal naive.